# JupyterLite 教育ベンチマーク（総合版 v4）

大学初年次 Python 入門講義向けの**全部入りベンチマーク**。v3 の計測項目（PyStack 各タイミング・WASM i32/f64 MOPS・5 カテゴリスコア）を**そのまま踏襲**し、過去データと比較可能なまま、次を追加・強化しました。

**v4 の改良点**
1. **結果ダッシュボード**（最後のセル）— 表＋レーダーチャート＋棒グラフを notebook 内に描画。JSON を読まずに一目で分かる。
2. **表計算へ即貼り付け**できる TSV 1 行を出力 — お手元のサマリー表にそのまま追記できる列順。
3. **ノイズ低減**— 計算系は warmup＋複数回測定の**中央値**、WASM は best-of-N。（Chrome で WASM が極端に低く出る等の単発アーティファクトを回避）
4. **失敗を空欄にしない**— 各計測を個別に try/except し、失敗は `null`＋理由を記録（弱端末で 1 項目こけても全体が止まらない）。
5. **WASM 層の判定を拡充**— SIMD / threads / crossOriginIsolated / bulk-memory。
6. **網羅性**— 純 Python ループ、FS スループット(MB/s)、（任意）SciPy を追加。
7. **弱端末向け Lite モード**— 先頭の CONFIG で重いセルを簡略化・反復数を調整可能。

| 柱 | 内容 | 取得 |
|---|---|---|
| A 初回体験 | 端末・ページロード・ネットワーク経路 | △（真の cold load は外部ハーネス） |
| B 信頼性 | ストレージ容量・FS 入出力(MB/s) | △（消失試験は手動） |
| C 実行時間 | import・計算・描画・WASM 層 | ○ |
| D | 総合スコア・ダッシュボード・エクスポート | ○ |

すべて Pyodide で完走するよう `try/except` 済み。最初のコードセルは `hello` を出力（自動ハーネス検出用）。
**使い方**: Run → **Run All Cells**。終わったら最後の表/グラフを確認し、TSV 行をコピー。

## ⚙️ CONFIG — まずここだけ編集（任意）

In [ ]:
# この端末・ブラウザのラベル（空なら自動判定）。例: "Mac Safari", "LIVA Firefox"
ENV_LABEL = ""
NOTES     = ""          # 自由メモ（回線条件など）。例: "cold cache / 学内WiFi"

# 弱端末向け設定
LITE      = False       # True: 重いセル(大きめ pandas/sympy)を簡略化
REPEATS   = 5           # 計算系の反復回数（中央値を採用）。弱端末では自動で減ります

# スコア基準（Apple M1 / Safari を 100 とする参照値。編集可）
BASELINE = {
    "python basics":        0.001,
    "import numpy":         0.226,
    "numpy ops":            0.013,
    "import pandas":        1.315,
    "read_csv + groupby":   0.023,
    "import sympy":         1.925,
    "sympy factorint":      0.031,
    "sympy expand":         0.127,
    "import matplotlib":    1.134,
    "line plot":            0.234,
    "histogram":            0.138,
    "ipywidgets interact":  0.005,
    "wasm_i32_mops":      200.0,    # これ以上で WASM=100
    "page_load_ms":      1500.0,    # これ以下で Load=100
    "fs_throughput_mbps":  50.0,    # これ以上で Storage=100
    "net_mbps":            50.0,    # これ以上で Network=100
}
# 総合スコアの重み（合計は内部で正規化）
WEIGHTS = {"Load": 0.25, "Network": 0.15, "PyStack": 0.35, "WASM": 0.15, "Storage": 0.10}

print("CONFIG OK — LITE =", LITE, "| REPEATS =", REPEATS)

In [ ]:
print("hello")

# Run All の起点付近で「ナビゲーション開始からの経過 ms」を記録（cold load + 初期化の概算プロキシ）
try:
    import js
    _PAGE_MS_AT_START = float(js.performance.now())
except Exception:
    _PAGE_MS_AT_START = None
_PAGE_MS_AT_START

## 0. 計測フックの準備

In [ ]:
import time, statistics as _stat

BENCH = []   # 実行時間の記録（label, seconds, ...）

class timer:
    """with timer("ラベル"): ... 単発計測（import など反復できないもの用）。失敗は null 記録。"""
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        if exc_type is None:
            BENCH.append({"label": self.label, "seconds": round(dt, 4), "samples": 1})
            print(f"[bench] {self.label}: {dt:.4f}s")
        else:
            BENCH.append({"label": self.label, "seconds": None, "error": repr(exc)})
            print(f"[bench][ERROR] {self.label}: {exc!r}")
        return True   # 例外を握りつぶして次セルへ

def bench_repeat(label, fn):
    """warmup + 複数回測定 → 中央値を記録（計算系のノイズ低減）。反復数は実行時間で自動調整。"""
    try:
        fn()  # warmup（JIT/キャッシュを温める）
        t0 = time.perf_counter(); fn(); first = time.perf_counter() - t0
        samples = [first]
        extra = (REPEATS - 1) if first < 2.0 else (2 if first < 8.0 else 0)
        for _ in range(extra):
            t0 = time.perf_counter(); fn(); samples.append(time.perf_counter() - t0)
        med = _stat.median(samples)
        BENCH.append({"label": label, "seconds": round(med, 4), "samples": len(samples),
                      "min": round(min(samples), 4), "max": round(max(samples), 4)})
        print(f"[bench] {label}: {med:.4f}s (median of {len(samples)})")
        return med
    except Exception as e:
        BENCH.append({"label": label, "seconds": None, "error": repr(e)})
        print(f"[bench][ERROR] {label}: {e!r}")
        return None

## A-1. 端末・ブラウザ情報

In [ ]:
ENV = {"label": ENV_LABEL or None, "user_agent": None, "browser": None, "os": None,
       "logical_cores": None, "device_memory_gb": None, "language": None, "notes": NOTES or None}

def _guess(ua):
    ua_l = (ua or "").lower(); b = o = None
    if "firefox" in ua_l: b = "Firefox"
    elif "edg" in ua_l: b = "Edge"
    elif "chrome" in ua_l or "crios" in ua_l: b = "Chrome"
    elif "safari" in ua_l: b = "Safari"
    if "windows" in ua_l: o = "Windows"
    elif "mac os" in ua_l or "macintosh" in ua_l: o = "macOS"
    elif "android" in ua_l: o = "Android"
    elif "iphone" in ua_l or "ipad" in ua_l: o = "iOS/iPadOS"
    elif "linux" in ua_l: o = "Linux"
    return b, o

try:
    import js
    nav = js.navigator
    ENV["user_agent"] = str(getattr(nav, "userAgent", "") or "") or None
    ENV["browser"], ENV["os"] = _guess(ENV["user_agent"])
    try: ENV["logical_cores"] = int(nav.hardwareConcurrency)
    except Exception: pass
    try: ENV["device_memory_gb"] = float(nav.deviceMemory)
    except Exception: pass
    try: ENV["language"] = str(nav.language)
    except Exception: pass
    if not ENV["label"] and ENV["browser"] and ENV["os"]:
        ENV["label"] = f"{ENV['os']} {ENV['browser']}"
except Exception as e:
    print(f"[env] ブラウザ API なし（通常カーネル？）: {e!r}")

for k, v in ENV.items():
    print(f"  {k:18}: {v}")
ENV

## A-2. ページロード時間 / cache 状態

ワーカ実行のため Navigation Timing が取れない場合は `performance.now()` プロキシにフォールバック。
> 真の cold load（初回 DL 込み）は外部ハーネス（Playwright 等）で測るのが正確。ここは warm/初期化の目安。

In [ ]:
PAGE_LOAD = {"page_ms_at_first_cell": _PAGE_MS_AT_START, "nav_total_ms": None,
             "dns_ms": None, "tcp_ms": None, "ttfb_ms": None, "transfer_size_bytes": None,
             "cache_state": None, "method": None}
try:
    import js
    navs = None
    try:
        navs = js.performance.getEntriesByType("navigation")
    except Exception:
        navs = None
    if navs is not None and navs.length and navs.length > 0:
        n0 = navs[0]
        PAGE_LOAD["method"] = "navigation-timing"
        PAGE_LOAD["nav_total_ms"] = round(float(n0.duration), 1)
        try: PAGE_LOAD["dns_ms"]  = round(float(n0.domainLookupEnd - n0.domainLookupStart), 1)
        except Exception: pass
        try: PAGE_LOAD["tcp_ms"]  = round(float(n0.connectEnd - n0.connectStart), 1)
        except Exception: pass
        try: PAGE_LOAD["ttfb_ms"] = round(float(n0.responseStart - n0.requestStart), 1)
        except Exception: pass
        try:
            ts = float(n0.transferSize); PAGE_LOAD["transfer_size_bytes"] = int(ts)
            PAGE_LOAD["cache_state"] = "warm(<=cached)" if ts == 0 else "cold/partial"
        except Exception: pass
    else:
        PAGE_LOAD["method"] = "performance.now-proxy"
    print(f"[load] method={PAGE_LOAD['method']} nav_total_ms={PAGE_LOAD['nav_total_ms']} "
          f"first_cell_ms={PAGE_LOAD['page_ms_at_first_cell']} cache={PAGE_LOAD['cache_state']}")
except Exception as e:
    print(f"[load] スキップ: {e!r}")
PAGE_LOAD

## A-3. ネットワーク経路

Resource Timing から最大アセットの実効スループット(Mbps)と接続経路を推定（取得できた範囲）。
ワーカからは見えるリソースが限られるため、取れない環境では `connection` API 等で代替。

In [ ]:
NETWORK = {"effective_mbps": None, "largest_asset_bytes": None, "largest_asset_ms": None,
           "downlink_mbps": None, "rtt_ms": None, "effective_type": None, "method": None}
try:
    import js
    # 1) Network Information API（取れれば手軽）
    try:
        conn = js.navigator.connection
        NETWORK["downlink_mbps"]   = float(conn.downlink)
        NETWORK["rtt_ms"]          = float(conn.rtt)
        NETWORK["effective_type"]  = str(conn.effectiveType)
    except Exception:
        pass
    # 2) Resource Timing から最大転送アセットの実効スループット
    try:
        res = js.performance.getEntriesByType("resource")
        best_sz, best_dur = 0, 0.0
        for i in range(int(res.length)):
            r = res[i]
            sz = float(getattr(r, "transferSize", 0) or 0)
            dur = float(r.duration or 0)
            if sz > best_sz and dur > 0:
                best_sz, best_dur = sz, dur
        if best_sz > 0:
            NETWORK["largest_asset_bytes"] = int(best_sz)
            NETWORK["largest_asset_ms"]    = round(best_dur, 1)
            NETWORK["effective_mbps"]      = round(best_sz * 8 / (best_dur/1000.0) / 1e6, 2)
            NETWORK["method"] = "resource-timing"
    except Exception:
        pass
    if NETWORK["effective_mbps"] is None and NETWORK["downlink_mbps"]:
        NETWORK["effective_mbps"] = NETWORK["downlink_mbps"]; NETWORK["method"] = "connection-api"
    print(f"[net] eff={NETWORK['effective_mbps']} Mbps downlink={NETWORK['downlink_mbps']} "
          f"rtt={NETWORK['rtt_ms']} type={NETWORK['effective_type']} method={NETWORK['method']}")
except Exception as e:
    print(f"[net] スキップ: {e!r}")
NETWORK

## B. ストレージ容量 / FS スループット

`navigator.storage.estimate()` で容量・永続化フラグ、仮想 FS への 4MB 書き込み→読み出しで MB/s を測定。
> タブ閉じ・別端末・キャッシュ消去による**データ消失**は notebook 内では検証不能 → 手動シナリオ試験へ。

In [ ]:
STORAGE = {"quota_mb": None, "usage_mb": None, "persisted": None,
           "fs_write_mbps": None, "fs_read_mbps": None, "fs_roundtrip_ms": None}
# 1) 容量見積り（async）
try:
    import js
    est = await js.navigator.storage.estimate()
    STORAGE["quota_mb"] = round(float(est.quota)/1e6, 1)
    STORAGE["usage_mb"] = round(float(est.usage)/1e6, 1)
    try: STORAGE["persisted"] = bool(await js.navigator.storage.persisted())
    except Exception: pass
except Exception as e:
    print(f"[storage] estimate スキップ: {e!r}")

# 2) FS スループット（4MB 書き込み/読み出し）
try:
    payload = (b"0123456789ABCDEF" * 64) * 4096   # 4 MiB
    mb = len(payload)/1e6
    t0 = time.perf_counter()
    with open("bench_io.bin", "wb") as f:
        f.write(payload)
    tw = time.perf_counter() - t0
    t0 = time.perf_counter()
    with open("bench_io.bin", "rb") as f:
        data = f.read()
    tr = time.perf_counter() - t0
    STORAGE["fs_write_mbps"]   = round(mb/tw, 1) if tw > 0 else None
    STORAGE["fs_read_mbps"]    = round(mb/tr, 1) if tr > 0 else None
    STORAGE["fs_roundtrip_ms"] = round((tw+tr)*1000, 1)
    import os
    try: os.remove("bench_io.bin")
    except Exception: pass
    print(f"[storage] write={STORAGE['fs_write_mbps']} read={STORAGE['fs_read_mbps']} MB/s "
          f"(roundtrip {STORAGE['fs_roundtrip_ms']} ms)")
except Exception as e:
    print(f"[storage] FS スキップ: {e!r}")
STORAGE

## C-1. Python 基礎（純 Python の素の速度）

In [ ]:
def _basics():
    # ループ・内包・関数呼び出し・素朴な素数判定（入門講義で書く典型コード）
    s = 0
    for i in range(200_000):
        s += i * i
    sq = [x*x for x in range(50_000)]
    def is_prime(n):
        if n < 2: return False
        i = 2
        while i*i <= n:
            if n % i == 0: return False
            i += 1
        return True
    primes = [n for n in range(2, 3000) if is_prime(n)]
    return s, len(sq), len(primes)

bench_repeat("python basics", _basics)

## C-2. NumPy（数値計算）

In [ ]:
with timer("import numpy"):
    import numpy as np

_rng = np.random.default_rng(0)
_A = _rng.standard_normal((400, 400))
_B = _rng.standard_normal((400, 400))

def _numpy_ops():
    C = _A @ _B                     # 行列積
    s = float(C.sum())
    v = float(np.linalg.norm(_A, axis=1).mean())
    return s, v

bench_repeat("numpy ops", _numpy_ops)

## C-3. pandas（表データ）

In [ ]:
with timer("import pandas"):
    import pandas as pd

import numpy as np, io
_nrows = 20_000 if LITE else 100_000
_df0 = pd.DataFrame({
    "g": np.random.default_rng(1).integers(0, 50, _nrows),
    "x": np.random.default_rng(2).standard_normal(_nrows),
    "y": np.random.default_rng(3).standard_normal(_nrows),
})
_csv = _df0.to_csv(index=False)

def _csv_groupby():
    df = pd.read_csv(io.StringIO(_csv))
    out = df.groupby("g").agg(x_mean=("x", "mean"), y_sum=("y", "sum"), n=("x", "size"))
    return out.shape

bench_repeat("read_csv + groupby", _csv_groupby)

## C-4. SymPy（記号計算）

In [ ]:
with timer("import sympy"):
    import sympy as sp

_M = 2**67 - 1 if not LITE else 2**59 - 1   # factorint の対象（LITE で軽く）
def _factor():
    return sp.factorint(_M)

x = sp.symbols("x")
_deg = 30 if not LITE else 18
def _expand():
    return sp.expand((x + 1)**_deg)

bench_repeat("sympy factorint", _factor)
bench_repeat("sympy expand", _expand)

## C-5. matplotlib（可視化）

In [ ]:
with timer("import matplotlib"):
    import matplotlib.pyplot as plt
import numpy as np

# 描画計測は draw()+close() で（出力をスパムせず、レンダリングコストのみ計測）
def _line():
    fig, ax = plt.subplots()
    ax.plot(range(200), [x**0.5 for x in range(200)], marker=".")
    ax.set_title("y = sqrt(x)")
    fig.canvas.draw(); plt.close(fig)

def _hist():
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots()
    ax.hist(rng.normal(size=5000), bins=40)
    ax.set_title("normal sample")
    fig.canvas.draw(); plt.close(fig)

bench_repeat("line plot", _line)
bench_repeat("histogram", _hist)

# 参考: サンプル図を1枚だけ表示（描画が実際に出るかの目視確認）
try:
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot(range(20), [x*x for x in range(20)], marker="o")
    ax.set_title("y = x^2 (sample)")
    plt.show()
except Exception:
    pass

## C-6. ipywidgets（対話ウィジェット）

In [ ]:
def _widgets():
    from ipywidgets import interact, IntSlider
    import io as _io
    from contextlib import redirect_stdout
    def f(k=3):
        print(k*k)
    buf = _io.StringIO()
    with redirect_stdout(buf):
        interact(f, k=IntSlider(min=0, max=10, value=3))
    return True

bench_repeat("ipywidgets interact", _widgets)

## C-7.（任意）SciPy — 入門でデータ解析に触れる場合

In [ ]:
try:
    import importlib.util
    if importlib.util.find_spec("scipy") is None:
        print("[scipy] 未インストール（micropip install scipy で追加可）— スキップ")
    else:
        with timer("import scipy"):
            import scipy
        import numpy as np
        from scipy import linalg, fft
        _S = np.random.default_rng(7).standard_normal((300, 300))
        _sig = np.random.default_rng(8).standard_normal(2**16)
        def _scipy_ops():
            x = linalg.solve(_S + 300*np.eye(300), np.ones(300))
            F = fft.rfft(_sig)
            return float(x.sum()), int(F.shape[0])
        bench_repeat("scipy ops", _scipy_ops)
except Exception as e:
    print(f"[scipy] スキップ: {e!r}")

## C-8. WASM 層スコア（純 WebAssembly の素の速度）

Python インタプリタを介さず、ブラウザの WASM エンジンで整数 / 浮動小数の単純ループを直接実行し Mops/s を測る。
**warmup + best-of-N** で単発の JIT 未温機アーティファクト（特定ブラウザで極端に低く出る現象）を回避。
あわせて SIMD・threads・crossOriginIsolated・bulk-memory を判定。
> 231 バイトの WASM モジュールを埋め込み（外部依存なし）。

In [ ]:
WASM = {"available": False, "i32_mops_per_s": None, "f64_mops_per_s": None,
        "simd_supported": None, "threads_capable": None,
        "cross_origin_isolated": None, "bulk_memory": None,
        "note": "Python 層を介さない WebAssembly エンジンの素の実行速度 (best-of-N)"}
WASM_B64 = "AGFzbQEAAAABCwJgAX8Bf2ABfwF8AwMCAAEHGQIJYmVuY2hfaTMyAAAJYmVuY2hfZjY0AAEKdQIxAQJ/QQEhAgJAA0AgASAATg0BIAJBjczlAGxB3+a74wNqIQIgAUEBaiEBDAALCyACC0ECAX8BfEQAAAAAAADwPyECAkADQCABIABODQEgAkTLGlDK///vP6JEmpmZmZmZuT+gIQIgAUEBaiEBDAALCyACCwA5BG5hbWUCGQIAAwABbgEBaQIDYWNjAQMAAW4BAWkCAXgDFwIAAgADYnJrAQJscAECAANicmsBAmxw"
try:
    import js, base64

    try:
        WASM["cross_origin_isolated"] = bool(getattr(js, "crossOriginIsolated", False))
    except Exception:
        pass
    try:
        WASM["threads_capable"] = bool(getattr(js, "SharedArrayBuffer", None)) and bool(WASM["cross_origin_isolated"])
    except Exception:
        pass
    # SIMD 検出（v128 を使う最小モジュールを validate）
    try:
        simd_probe = bytes([0,97,115,109,1,0,0,0,1,5,1,96,0,1,123,3,2,1,0,10,10,1,8,0,65,0,253,15,253,98,11])
        sp = js.Uint8Array.new(len(simd_probe)); sp.assign(bytearray(simd_probe))
        WASM["simd_supported"] = bool(js.WebAssembly.validate(sp))
    except Exception:
        pass
    # bulk-memory 検出（memory.fill を使う最小モジュールを validate）
    try:
        bm_probe = bytes([0,97,115,109,1,0,0,0,1,4,1,96,0,0,3,2,1,0,5,3,1,0,1,10,11,1,9,0,65,0,65,0,65,0,252,11,0,11])
        bp = js.Uint8Array.new(len(bm_probe)); bp.assign(bytearray(bm_probe))
        WASM["bulk_memory"] = bool(js.WebAssembly.validate(bp))
    except Exception:
        pass

    raw = base64.b64decode(WASM_B64)
    u8 = js.Uint8Array.new(len(raw)); u8.assign(bytearray(raw))
    res = await js.WebAssembly.instantiate(u8, js.Object.new())
    exports = res.instance.exports
    N = 50_000_000

    def best_mops(fn, n, rounds=5):
        fn(min(n, 5_000_000))          # warmup
        best = 0.0
        for _ in range(rounds):
            t0 = js.performance.now(); fn(n); dt = (js.performance.now() - t0)/1000.0
            if dt > 0: best = max(best, n/dt/1e6)
        return round(best, 1)

    WASM["i32_mops_per_s"] = best_mops(exports.bench_i32, N)
    WASM["f64_mops_per_s"] = best_mops(exports.bench_f64, N)
    WASM["available"] = True
    print(f"[wasm] i32={WASM['i32_mops_per_s']} f64={WASM['f64_mops_per_s']} Mops/s | "
          f"simd={WASM['simd_supported']} threads={WASM['threads_capable']} "
          f"COI={WASM['cross_origin_isolated']} bulkmem={WASM['bulk_memory']}")
except Exception as e:
    print(f"[wasm] スキップ/失敗: {e!r}")
WASM

## D-1. スコア集計（0–100、機種横断で比較可能）

各計測を基準機（M1/Safari=100）と比べ、**速いほど高得点**（100 で頭打ち）。
カテゴリは各項目スコアの**幾何平均**、総合は重み付き幾何平均。基準・重みは CONFIG で変更可。

In [ ]:
import math

def _score_lower_better(measured, base):
    if measured is None or measured <= 0: return None
    return max(0.0, min(100.0, 100.0 * base / measured))

def _score_higher_better(measured, base):
    if measured is None or measured <= 0: return None
    return max(0.0, min(100.0, 100.0 * measured / base))

def _geomean(vals):
    vals = [v for v in vals if v is not None and v > 0]
    if not vals: return None
    return round(math.exp(sum(math.log(v) for v in vals)/len(vals)), 1)

# BENCH を label -> seconds 辞書に
_secs = {b["label"]: b.get("seconds") for b in BENCH}

# PyStack: 12 項目の各スコアの幾何平均（v3 と同じ集計）
_pystack_labels = ["python basics","import numpy","numpy ops","import pandas","read_csv + groupby",
                   "import sympy","sympy factorint","sympy expand","import matplotlib",
                   "line plot","histogram","ipywidgets interact"]
_pystack_scores = {lab: _score_lower_better(_secs.get(lab), BASELINE[lab]) for lab in _pystack_labels}

SCORES = {
    "Load":    _score_lower_better(PAGE_LOAD.get("page_ms_at_first_cell"), BASELINE["page_load_ms"]),
    "Network": _score_higher_better(NETWORK.get("effective_mbps"), BASELINE["net_mbps"]),
    "PyStack": _geomean(list(_pystack_scores.values())),
    "WASM":    _score_higher_better(WASM.get("i32_mops_per_s"), BASELINE["wasm_i32_mops"]),
    "Storage": _score_higher_better(STORAGE.get("fs_write_mbps"), BASELINE["fs_throughput_mbps"]),
}

# 総合 = 重み付き幾何平均（取得できたカテゴリのみで正規化）
_pairs = [(SCORES[k], WEIGHTS[k]) for k in WEIGHTS if SCORES.get(k) is not None and SCORES[k] > 0]
if _pairs:
    _wsum = sum(w for _, w in _pairs)
    COMPOSITE = round(math.exp(sum(w*math.log(s) for s, w in _pairs)/_wsum), 1)
else:
    COMPOSITE = None

print("Composite:", COMPOSITE)
for k in ["Load","Network","PyStack","WASM","Storage"]:
    print(f"  {k:8}: {SCORES[k]}")
SCORES["_composite"] = COMPOSITE
SCORES

## D-2. 結果ダッシュボード（表・レーダー・棒グラフ）

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- (1) サマリー表 ---
try:
    import pandas as pd
    rows = [("Composite", COMPOSITE)] + [(k, SCORES[k]) for k in ["Load","Network","PyStack","WASM","Storage"]]
    disp = pd.DataFrame(rows, columns=["Category", "Score (0-100)"])
    print(f"=== {ENV.get('label') or 'this environment'} ===")
    try:
        from IPython.display import display
        display(disp)
    except Exception:
        print(disp.to_string(index=False))
except Exception as e:
    print("[table]", repr(e))

# --- (2) レーダーチャート（5 カテゴリ） ---
try:
    cats = ["Load","Network","PyStack","WASM","Storage"]
    vals = [SCORES.get(c) or 0 for c in cats]
    ang = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
    vals2 = vals + vals[:1]; ang2 = ang + ang[:1]
    fig = plt.figure(figsize=(5,5)); ax = plt.subplot(111, polar=True)
    ax.plot(ang2, vals2, linewidth=2); ax.fill(ang2, vals2, alpha=0.25)
    ax.set_xticks(ang); ax.set_xticklabels(cats)
    ax.set_ylim(0, 100); ax.set_title(f"Category scores  (Composite={COMPOSITE})")
    plt.show()
except Exception as e:
    print("[radar]", repr(e))

# --- (3) 実行時間の棒グラフ（対数軸） ---
try:
    labs = ["python basics","import numpy","numpy ops","import pandas","read_csv + groupby",
            "import sympy","sympy factorint","sympy expand","import matplotlib",
            "line plot","histogram","ipywidgets interact"]
    pairs = [(l, _secs.get(l)) for l in labs if _secs.get(l) is not None and _secs.get(l) > 0]
    if pairs:
        names = [p[0] for p in pairs]; times = [p[1] for p in pairs]
        fig, ax = plt.subplots(figsize=(7, 4.5))
        y = np.arange(len(names)); ax.barh(y, times)
        ax.set_yticks(y); ax.set_yticklabels(names); ax.invert_yaxis()
        ax.set_xscale("log"); ax.set_xlabel("seconds (log)")
        ax.set_title("Execution time per task")
        for i, t in enumerate(times):
            ax.text(t, i, f" {t:.3f}", va="center", fontsize=8)
        plt.tight_layout(); plt.show()
except Exception as e:
    print("[bar]", repr(e))

## D-3. エクスポート

`bench_results.json`（schema v4・全フィールド）に保存し、**表計算へ即貼り付けできる TSV 1 行**も出力します。
端末×ブラウザごとにこの 1 行を集めれば、お手元のサマリー表がそのまま埋まります。

In [ ]:
import json

payload = {
    "schema": "v4",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "composite": COMPOSITE,
    "scores": {k: SCORES[k] for k in ["Load","Network","PyStack","WASM","Storage"]},
    "environment": ENV,
    "page_load": PAGE_LOAD,
    "network": NETWORK,
    "storage": STORAGE,
    "wasm": WASM,
    "measurements": BENCH,
    "config": {"LITE": LITE, "REPEATS": REPEATS, "BASELINE": BASELINE, "WEIGHTS": WEIGHTS},
}
text = json.dumps(payload, ensure_ascii=False, indent=2)
try:
    with open("bench_results.json", "w") as f:
        f.write(text)
    print("[saved] bench_results.json （ファイルブラウザからDL可）\n")
except Exception as e:
    print(f"[warn] 保存失敗: {e!r}\n")

# --- 表計算へ貼り付け用 TSV（既存サマリー表と同じ列順） ---
def g(lab): 
    v = _secs.get(lab)
    return "" if v is None else f"{v:.3f}"
def gs(k):
    v = SCORES.get(k) if k != "Composite" else COMPOSITE
    return "" if v is None else f"{v:.3f}"

osc = ENV.get("os") or ""
cores = ENV.get("logical_cores"); memg = ENV.get("device_memory_gb")
oscpumem = " / ".join([s for s in [osc,
                                   (f"{cores}-core" if cores else ""),
                                   (f"{memg}GB" if memg else "")] if s])

header = ["Environment","OS / CPU / Memory","Composite Score","Load","Network","PyStack","WASM","Storage",
          "WASM i32 MOPS","Import pandas (s)","Import sympy (s)",
          "python basics (s)","import numpy (s)","numpy ops (s)","import pandas (s)","read_csv + groupby (s)",
          "import sympy (s)","sympy factorint (s)","sympy expand (s)","import matplotlib (s)",
          "line plot (s)","histogram (s)","ipywidgets interact (s)"]
row = [ENV.get("label") or "", oscpumem, gs("Composite"), gs("Load"), gs("Network"), gs("PyStack"),
       gs("WASM"), gs("Storage"),
       (f"{WASM.get('i32_mops_per_s'):.1f}" if WASM.get("i32_mops_per_s") else ""),
       g("import pandas"), g("import sympy"),
       g("python basics"), g("import numpy"), g("numpy ops"), g("import pandas"), g("read_csv + groupby"),
       g("import sympy"), g("sympy factorint"), g("sympy expand"), g("import matplotlib"),
       g("line plot"), g("histogram"), g("ipywidgets interact")]

print("===== 表計算へ貼り付け（タブ区切り。1行目ヘッダは初回のみ）=====")
print("\t".join(header))
print("\t".join(row))